in this version we have to download data after each session and rename it like P01_smell.txt OR P01_no_smell.txt in a folder: experiment_data/ and then zip it
   

    

#Import files

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Fake zip.zip to Fake zip.zip


# Extract Zip file

In [ ]:
import zipfile
import os
import glob
import pandas as pd
import numpy as np

zip_filename = list(uploaded.keys())[0]
extract_folder = "experiment_data"

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Files extracted successfully.")

Files extracted successfully.


#Calculate SART2 primary outcomes

In [ ]:
# Find all txt files inside the extracted folder
all_txt_files = glob.glob(
    "experiment_data/**/*.txt",
    recursive=True
)

results = []

for filepath in all_txt_files:

    filename = os.path.basename(filepath)
    name = filename.replace(".txt", "")

    # Expected file names:
    # P01_smell.txt
    # P01_no_smell.txt
    participant, condition = name.split("_", 1)

    # Read PsyToolkit data
    data = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None
    )

    data.columns = [
        "block",
        "blocknum",
        "go_nogo",
        "digit",
        "stimulus",
        "correct",
        "rt"
    ]

    # Keep only real test trials
    data = data[data["block"] == "realtest"]

    if data.empty:
        print("No realtest trials found:", filename)
        continue

    # Define trial types
    go_trials = data[data["go_nogo"] == 1]
    nogo_trials = data[data["go_nogo"] == 0]

    # Commission errors:
    # Incorrect responses on no-go trials
    commission_errors = len(
        nogo_trials[nogo_trials["correct"] == 0]
    )

    commission_error_percent = (
        commission_errors / len(nogo_trials)
    ) * 100

    # Omission errors:
    # Failure to respond on go trials
    omission_errors = len(
        go_trials[go_trials["correct"] == 0]
    )

    omission_error_percent = (
        omission_errors / len(go_trials)
    ) * 100

    # RT variability:
    # Standard deviation of RT from correct go trials only
    # Implausible RTs below 100 ms or above 1500 ms are excluded
    correct_go = go_trials[
        (go_trials["correct"] == 1) &
        (go_trials["rt"] >= 100) &
        (go_trials["rt"] <= 1500)
    ]

    rt_variability = correct_go["rt"].std()

    results.append({
        "ParticipantID": participant,
        "Condition": condition,
        "CommissionErrors": commission_error_percent,
        "OmissionErrors": omission_error_percent,
        "RTVariability": rt_variability
    })

# Create long-format dataframe
results_df = pd.DataFrame(results)

print("Long-format results:")
display(results_df)

# Convert to wide format: one row per participant
wide_df = results_df.pivot(
    index="ParticipantID",
    columns="Condition"
)

# Flatten column names
wide_df.columns = [
    f"{col[0]}_{col[1]}"
    for col in wide_df.columns
]

wide_df = wide_df.reset_index()

# Reorder and rename columns
wide_df = wide_df[[
    "ParticipantID",

    "CommissionErrors_no_smell",
    "CommissionErrors_smell",

    "OmissionErrors_no_smell",
    "OmissionErrors_smell",

    "RTVariability_no_smell",
    "RTVariability_smell"
]]

wide_df.columns = [
    "ParticipantID",

    "CommissionErrors_NoSmell",
    "CommissionErrors_Smell",

    "OmissionErrors_NoSmell",
    "OmissionErrors_Smell",

    "RTVariability_NoSmell",
    "RTVariability_Smell"
]

# Sort participants
wide_df = wide_df.sort_values("ParticipantID")

print("Final participant-level SART2 primary outcomes:")
display(wide_df)

# Save and download primary outcome dataset
primary_outcomes_file = "SART2_PrimaryOutcomes.xlsx"

wide_df.to_excel(
    primary_outcomes_file,
    index=False
)

Long-format results:


,ParticipantID,Condition,CommissionErrors,OmissionErrors,RTVariability
0,P01,no_smell,33.333333,0.000000,128.521073
1,P01,smell,16.666667,2.083333,211.589404
2,P02,smell,33.333333,4.166667,218.580724
3,P02,no_smell,33.333333,2.083333,137.063717


Final participant-level SART2 primary outcomes:


,ParticipantID,CommissionErrors_NoSmell,CommissionErrors_Smell,OmissionErrors_NoSmell,OmissionErrors_Smell,RTVariability_NoSmell,RTVariability_Smell
0,P01,33.333333,16.666667,0.000000,2.083333,128.521073,211.589404
1,P02,33.333333,33.333333,2.083333,4.166667,137.063717,218.580724


#Descriptive statistics table

In [ ]:
descriptive_results = []

measures = {
    "Commission Error Percentage": {
        "No-Smell": "CommissionErrors_NoSmell",
        "Smell": "CommissionErrors_Smell"
    },
    "Omission Error Percentage": {
        "No-Smell": "OmissionErrors_NoSmell",
        "Smell": "OmissionErrors_Smell"
    },
    "RT Variability": {
        "No-Smell": "RTVariability_NoSmell",
        "Smell": "RTVariability_Smell"
    }
}

for measure_name, conditions in measures.items():
    for condition_name, column_name in conditions.items():

        descriptive_results.append({
            "Measure": measure_name,
            "Condition": condition_name,
            "Mean": wide_df[column_name].mean(),
            "SD": wide_df[column_name].std()
        })

descriptive_df = pd.DataFrame(descriptive_results)

display(descriptive_df)

descriptive_file = "SART2_DescriptiveStatistics.xlsx" #Table 5.2

descriptive_df.to_excel(
    descriptive_file,
    index=False
)

,Measure,Condition,Mean,SD
0,Commission Error Percentage,No-Smell,33.333333,0.000000
1,Commission Error Percentage,Smell,25.000000,11.785113
2,Omission Error Percentage,No-Smell,1.041667,1.473139
3,Omission Error Percentage,Smell,3.125000,1.473139
4,RT Variability,No-Smell,132.792395,6.040561
5,RT Variability,Smell,215.085064,4.943610


#Shapiro–Wilk normality test for paired difference scores

In [ ]:
from scipy.stats import shapiro

normality_results = []

comparisons = {
    "Commission Error Percentage": {
        "Smell": "CommissionErrors_Smell",
        "No-Smell": "CommissionErrors_NoSmell"
    },
    "Omission Error Percentage": {
        "Smell": "OmissionErrors_Smell",
        "No-Smell": "OmissionErrors_NoSmell"
    },
    "RT Variability": {
        "Smell": "RTVariability_Smell",
        "No-Smell": "RTVariability_NoSmell"
    }
}

for measure_name, cols in comparisons.items():

    difference_scores = (
        wide_df[cols["Smell"]] - wide_df[cols["No-Smell"]]
    ).dropna()

    if len(difference_scores) >= 3:
        stat, p_value = shapiro(difference_scores)

        normality_results.append({
            "Measure": measure_name,
            "Shapiro_W": stat,
            "p_value": p_value,
            "Normality_Assumption": "Met" if p_value >= 0.05 else "Not met"
        })

    else:
        normality_results.append({
            "Measure": measure_name,
            "Shapiro_W": np.nan,
            "p_value": np.nan,
            "Normality_Assumption": "Not enough data"
        })

normality_df = pd.DataFrame(normality_results)

display(normality_df)

normality_file = "SART2_NormalityResults.xlsx" #Table F.1

normality_df.to_excel(
    normality_file,
    index=False
)

,Measure,Shapiro_W,p_value,Normality_Assumption
0,Commission Error Percentage,NaN,NaN,Not enough data
1,Omission Error Percentage,NaN,NaN,Not enough data
2,RT Variability,NaN,NaN,Not enough data


#Paired t-test

In [ ]:
from scipy.stats import ttest_rel

ttest_results = []

for measure_name, cols in comparisons.items():

    paired_data = wide_df[
        ["ParticipantID", cols["Smell"], cols["No-Smell"]]
    ].dropna()

    smell_values = paired_data[cols["Smell"]]
    no_smell_values = paired_data[cols["No-Smell"]]

    difference_scores = smell_values - no_smell_values

    if len(paired_data) >= 2:

        t_stat, p_value = ttest_rel(
            smell_values,
            no_smell_values
        )

        diff_sd = difference_scores.std(ddof=1)

        if diff_sd == 0:
            cohens_d = np.nan
        else:
            cohens_d = difference_scores.mean() / diff_sd

        ttest_results.append({
            "Measure": measure_name,
            "N": len(paired_data),
            "Smell_Mean": smell_values.mean(),
            "Smell_SD": smell_values.std(ddof=1),
            "NoSmell_Mean": no_smell_values.mean(),
            "NoSmell_SD": no_smell_values.std(ddof=1),
            "Mean_Difference_Smell_minus_NoSmell": difference_scores.mean(),
            "t_value": t_stat,
            "p_value": p_value,
            "Cohens_d": cohens_d
        })

    else:

        ttest_results.append({
            "Measure": measure_name,
            "N": len(paired_data),
            "Smell_Mean": np.nan,
            "Smell_SD": np.nan,
            "NoSmell_Mean": np.nan,
            "NoSmell_SD": np.nan,
            "Mean_Difference_Smell_minus_NoSmell": np.nan,
            "t_value": np.nan,
            "p_value": np.nan,
            "Cohens_d": np.nan
        })

ttest_results_df = pd.DataFrame(ttest_results)

display(ttest_results_df)

ttest_file = "SART2_PairedTTestResults.xlsx" #Table 5.3

ttest_results_df.to_excel(
    ttest_file,
    index=False
)

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)


,Measure,N,Smell_Mean,Smell_SD,NoSmell_Mean,NoSmell_SD,Mean_Difference_Smell_minus_NoSmell,t_value,p_value,Cohens_d
0,Commission Error Percentage,2,25.000000,11.785113,33.333333,0.000000,-8.333333,-1.000000,0.500,-0.707107
1,Omission Error Percentage,2,3.125000,1.473139,1.041667,1.473139,2.083333,inf,0.000,NaN
2,RT Variability,2,215.085064,4.943610,132.792395,6.040561,82.292669,106.093461,0.006,75.019406


#Downloading Results

In [ ]:
from google.colab import files

files.download("SART2_PrimaryOutcomes.xlsx")              #needed for master dataset
files.download("SART2_DescriptiveStatistics.xlsx")        #Table 5.2
files.download("SART2_NormalityResults.xlsx")             #Table F.1
files.download("SART2_PairedTTestResults.xlsx")           #Table 5.3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>